# v11 Formalizer Fine-Tune (NL → FOL string, GOLD-supervised)
**EXACT 2026 — Qwen3-8B + Unsloth QLoRA | Kaggle T4×2**

Khác hẳn v6 (fine-tune trả lời → giết reasoning, 33.3%) và v10 (STaR silver).
Bản này train trên **gold thật**: dataset có `premises-NL` ↔ `premises-FOL` song song
(≈4337 cặp). Task hẹp & rõ: dịch NL → FOL string.

**Mục đích:** phục vụ `PREMISE_SOURCE="lora_fol"` trong v11 inference — khi test set CHỈ
có NL (không kèm gold FOL). LoRA sinh FOL string → parser deterministic → Z3.
Dùng LoRA CHỈ cho dịch FOL; model gốc giữ cho reasoning fallback.

Format khớp `FOL_TRANSLATE_SYSTEM` của v11 inference: input = premise NL đánh số,
output = mỗi dòng `i: <FOL>`.


In [ ]:
#!/usr/bin/env python3
"""
notebook_v11_formalizer_finetune.py
GOLD-supervised NL->FOL translator LoRA (Qwen3-8B + Unsloth QLoRA).
Train on premises-NL <-> premises-FOL pairs from the challenge dataset.
Output: /kaggle/working/qwen3_fol_translator_lora  (PEFT adapter, vLLM-compatible)
Use with v11 inference: PREMISE_SOURCE="lora_fol", FORMALIZER_LORA_PATH=<this dir>.
"""


In [ ]:
# ================= Kaggle T4 fix =================
import os, shutil, glob
STUB_DIR="/tmp/cuda_stubs"; os.makedirs(STUB_DIR,exist_ok=True)
stub=os.path.join(STUB_DIR,"libcuda.so")
if os.path.exists(stub) or os.path.islink(stub): os.remove(stub)
for c in ["/usr/lib/x86_64-linux-gnu/libcuda.so.1","/usr/lib/x86_64-linux-gnu/libcuda.so",
          *glob.glob("/usr/**/libcuda.so*",recursive=True)]:
    if os.path.exists(c) and not os.path.islink(c): os.symlink(c,stub); break
os.environ["LIBRARY_PATH"]=f"{STUB_DIR}:"+os.environ.get("LIBRARY_PATH","")
os.environ["LD_LIBRARY_PATH"]=f"{STUB_DIR}:"+os.environ.get("LD_LIBRARY_PATH","")
shutil.rmtree("/root/.cache/flashinfer",ignore_errors=True)
print("Kaggle T4 fixes applied!")


In [ ]:
# ================= INSTALL =================
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","--quiet","--break-system-packages",
                "unsloth","trl<=0.24.0","datasets<4.4.0"],check=True)
subprocess.run([sys.executable,"-m","pip","install","--quiet","--upgrade","--break-system-packages",
                "--no-cache-dir","peft","accelerate","bitsandbytes"],check=True)
import torch; print("PyTorch",torch.__version__,"CUDA",torch.cuda.is_available()); print("Install OK")


In [ ]:
# ================= CONFIG =================
MODEL_PATH     = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True

DATASET_PATH   = "/kaggle/input/logic-based-educational-queries/Logic_Based_Educational_Queries (2).json"
SEED           = 42
TRAIN_RATIO    = 0.85   # train on TRAIN split only (hold out for honest eval)
VAL_RATIO      = 0.10
GRANULARITY    = "sample"   # "sample" (numbered block, matches inference) | "premise" (1 pair each)

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

EPOCHS=3; BATCH_SIZE=2; GRAD_ACCUM=4; LEARNING_RATE=1e-4
WARMUP_STEPS=10; WEIGHT_DECAY=0.01; OPTIMIZER="adamw_8bit"; LR_SCHEDULER="linear"
LOGGING_STEPS=10
ENABLE_THINKING_TRAIN=False   # must match inference (FOL translate = no_think)

LORA_OUTPUT_DIR="/kaggle/working/qwen3_fol_translator_lora"
CHECKPOINT_DIR ="/kaggle/working/fol_translator_ckpt"

# MUST match FOL_TRANSLATE_SYSTEM in v11 inference notebook
FOL_TRANSLATE_SYSTEM = (
    "You are a First-Order Logic translator. Translate EACH numbered natural-language "
    "premise into ONE FOL formula. Use ONLY these symbols: forall as the unicode FOR ALL, "
    "exists, ->, <->, not, and, or, and Predicate(x) atoms. PascalCase predicate names, "
    "lowercase variables. Output exactly one formula per line, prefixed 'i: ', no commentary."
)
print("Config OK")


In [ ]:
# ================= LOAD MODEL =================
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=LOAD_IN_4BIT)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED)
trn=sum(p.numel() for p in model.parameters() if p.requires_grad)
tot=sum(p.numel() for p in model.parameters())
print(f"Trainable: {trn:,}/{tot:,} ({100*trn/tot:.2f}%)"); print("Model+LoRA ready")


In [ ]:
# ================= BUILD GOLD NL->FOL DATASET =================
import json, random
from datasets import Dataset

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)
rng = random.Random(SEED); rng.shuffle(raw)
n=len(raw); n_tr=int(n*TRAIN_RATIO); n_va=int(n*VAL_RATIO)
train_raw = raw[:n_tr]
print(f"Samples: {n} | train: {len(train_raw)}")

def make_user(nls):
    body="\n".join(f"{i+1}: {p}" for i,p in enumerate(nls))
    return f"Translate each premise to FOL:\n{body}"

records=[]
skipped=0
for s in train_raw:
    nls = s.get("premises-NL", []); fols = s.get("premises-FOL", [])
    if not nls or len(nls)!=len(fols):
        skipped+=1; continue
    if GRANULARITY=="sample":
        target="\n".join(f"{i+1}: {fols[i]}" for i in range(len(fols)))
        records.append({"conversations":[
            {"role":"system","content":FOL_TRANSLATE_SYSTEM},
            {"role":"user","content":make_user(nls)},
            {"role":"assistant","content":target},
        ]})
    else:  # premise-level
        for nl, fol in zip(nls, fols):
            records.append({"conversations":[
                {"role":"system","content":FOL_TRANSLATE_SYSTEM},
                {"role":"user","content":f"Translate to FOL:\n1: {nl}"},
                {"role":"assistant","content":f"1: {fol}"},
            ]})
print(f"Training examples: {len(records)} (skipped {skipped} mismatched samples)")
train_ds = Dataset.from_list(records)
print("Sample target preview:\n", records[0]["conversations"][-1]["content"][:200])


In [ ]:
# ================= TRAINER (responses-only) =================
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def formatting_func(examples):
    convos = examples["conversations"]
    def _fmt(c):
        try:
            return tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False,
                                                 enable_thinking=ENABLE_THINKING_TRAIN)
        except TypeError:
            return tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
    if len(convos)>0 and isinstance(convos[0], dict): return [_fmt(convos)]
    return [_fmt(c) for c in convos]

sft = SFTConfig(
    output_dir=CHECKPOINT_DIR, per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER, lr_scheduler_type=LR_SCHEDULER, logging_steps=LOGGING_STEPS,
    save_strategy="epoch", save_total_limit=1, seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH, dataset_text_field="text", packing=False, report_to="none")

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                     args=sft, formatting_func=formatting_func)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
print("Trainer ready | examples:", len(train_ds), "| eff.batch:", BATCH_SIZE*GRAD_ACCUM)


In [ ]:
# ================= TRAIN + SAVE =================
import time, os
t0=time.time(); res=trainer.train()
print(f"Done {(time.time()-t0)/60:.1f} min | loss {res.training_loss:.4f}")
model.save_pretrained(LORA_OUTPUT_DIR); tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print("Saved:", LORA_OUTPUT_DIR, "|", os.listdir(LORA_OUTPUT_DIR))
print()
print("NEXT (v11 inference, NL-only scenario):")
print('  PREMISE_SOURCE      = "lora_fol"')
print(f'  FORMALIZER_LORA_PATH = "{LORA_OUTPUT_DIR}"')


In [ ]:
# ================= QUICK SANITY (optional) =================
FastLanguageModel.for_inference(model)
msgs=[{"role":"system","content":FOL_TRANSLATE_SYSTEM},
      {"role":"user","content":"Translate each premise to FOL:\n1: All students must enroll in at least one core subject.\n2: If a student attends all lectures, they will have a higher chance of passing."}]
ids=tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out=model.generate(input_ids=ids, max_new_tokens=256, temperature=0.1, do_sample=True)
print(tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True))
